# Fidelidade da porta CPHASE com molas em estado térmico (aproximação de mola lenta)

Este notebook explica `calcular_termico.py` e `plot_termico.py`. Para cada variável e expressão ele mostra a fórmula e diz **qual função do código a implementa**. As células importam `calcular_termico.py`, mas **não** sobrescrevem os `.npz` nem a figura.

**O sistema** é o mesmo de `calcular_fock.py`: um sítio de Brod-Combes com dois átomos $a$ e $b$, acoplados por cross-Kerr $\chi$. Cada átomo está acoplado a uma "mola" de frequência $\omega^{(a,b)}$ e acoplamento $g^{(a,b)}$. A diferença é que agora as molas começam no **estado térmico**
$$\rho_{\rm th}=\sum_n p_n|n\rangle\langle n|,\qquad p_n=\frac{\bar n^{\,n}}{(1+\bar n)^{n+1}} .$$

| arquivo | o que faz |
|---|---|
| `calcular_termico.py` | calcula $F_1(\sigma)$, $F_2(\sigma)$ na aproximação de mola lenta e salva em `data/termico_*.npz` |
| `plot_termico.py` | desenha `fidelidade_sigma_mola_termico.svg/.png` no estilo da Fig. 5 do artigo |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import calcular_termico as c

## 1. Por que não fazer o cálculo térmico exato

A fidelidade $F=\langle\psi_{\rm id}|\rho_{\rm fótons}|\psi_{\rm id}\rangle$ é **linear** no estado inicial da mola, e $\rho_{\rm th}$ é diagonal na base de Fock. Por isso o resultado exato é
$$F_{\rm térmico}=\sum_{n_a,n_b}p_{n_a}\,p_{n_b}\;F_{\rm Fock}(n_a,n_b),$$
em que cada $F_{\rm Fock}$ é o cálculo completo de `calcular_fock.py`. Com $\bar n=1$ são necessários ~10 níveis por mola, ou seja ~100 pares $(n_a,n_b)$ **por valor de $\sigma$**. Para $\bar n=10$ seriam milhares. Uma varredura de 100 valores de $\sigma$ levaria horas.

Essa soma exata foi mantida em **`fidelidades_exato`**, que reaproveita `calcular_fock.py` e serve só para **validar** a aproximação em poucos pontos (seção 7). Um detalhe importante dela: o fóton ideal de referência precisa ser **o mesmo para todos os $n$**, porque ele não "conhece" o estado da mola. Por isso a fase de referência é a da amplitude elástica média térmica, $\sum_n p_n t_{nn}(\nu)$, e não a de cada $t_{nn}$ separadamente.

## 2. A aproximação de mola lenta ($\omega^{(a,b)}\ll\gamma$)

A estrutura de $S$ ($\Gamma_{kn}$ com o deslocamento $-4g^2/\omega$ e $F_{kn}=\langle k|D(2g/\omega)|n\rangle$) corresponde a uma mola $\omega\,b^\dagger b$ acoplada ao estado excitado do átomo por $2g\,(b+b^\dagger)$.

Se a mola é lenta comparada com o tempo de resposta do átomo ($1/\gamma$), ela não se move enquanto o fóton interage. Nesse caso o operador $\hat x=b+b^\dagger$ é conservado, e o átomo vê apenas um **desvio estático da linha**
$$\delta=2g\,x .$$
O operador de espalhamento fica diagonal em $x$:
$$U=\int dx\;|x\rangle\langle x|\otimes U_{\delta(x)},$$
em que $U_\delta$ é o sítio de Brod-Combes **sem mola** com a dessintonia $\Delta\to\Delta+\delta$. Ao traçar a mola,
$$\rho_{\rm fótons}=\int dx\,\langle x|\rho_{\rm th}|x\rangle\;U_{\delta(x)}|\psi\rangle\langle\psi|U_{\delta(x)}^\dagger .$$
O estado misto vira uma **mistura clássica** de sítios sem mola, cada um com a linha deslocada.

**Distribuição de $\delta$.** No estado térmico, $\langle x|\rho_{\rm th}|x\rangle$ é gaussiana com $\langle x^2\rangle=2\bar n+1$, logo
$$\delta\sim\mathcal N(0,s^2),\qquad s^2=4g^2(2\bar n+1)=S\,\omega^2(2\bar n+1),\qquad S=(2g/\omega)^2 .$$
Isso bate com o caso exato. As linhas de $t_{mn}$ ficam em $\Delta-4g^2/\omega+(k-n)\omega$ com pesos $F_{kn}^2$, cuja média é $\Delta$ (o deslocamento polarônico se cancela) e cuja variância é $4g^2(2n+1)$.

**Validade.** O átomo precisa responder antes de a mola se mover ($\omega\ll\gamma$). Além disso, as bandas laterais de $\pm\omega$ não devem ser resolvidas pelo pacote ($\omega\lesssim\sigma$). Quando $\sigma\ll\omega$, o fóton distingue espectralmente a troca de quanta com a mola e a aproximação perde precisão (seção 7). `calcular_caso` avisa se $\omega>0.2\gamma$.

**Consequência:** na aproximação lenta só importa a largura $s$. Os parâmetros $\omega$, $S$ e $\bar n$ entram apenas pela combinação $s=\omega\sqrt{S(2\bar n+1)}$ (**`largura_desvio`**).

In [ ]:
for nbar in [0, 2, 10]:
    print(f'nbar = {nbar:2d}:  s = {c.largura_desvio(0.3, 4.0, nbar):.3f}   (gamma = {c.gam})')

## 3. Parâmetros (topo de `calcular_termico.py`)

| variável | símbolo | significado |
|---|---|---|
| `chi`, `gam`, `Delta`, `omega0`, `phi` | $\chi,\gamma,\Delta,\omega_0,\phi$ | sítio e porta, iguais aos da Fig. 5 ($5,\,4.5,\,0,\,1.1,\,\pi$) |
| `omega_c` | $\omega_c=\Delta+\omega_0$ | centro do pacote |
| `referencia` | — | fóton ideal: `'media'` (fase de $\langle t_\delta\rangle$) ou `'sem_mola'` ($-\Gamma^*/\Gamma$) |
| `sigma_min`, `sigma_max`, `n_sigma` | $\sigma$ | varredura log |
| `k_sigma`, `pts_per_width`, `n_min`, `n_max` | — | grade de frequências (igual a `calcular_fock.py`) |
| `n_gh` | — | nós de Gauss-Hermite na média sobre $\delta$ |
| `tol_pop` | — | cauda térmica desprezada (só em `fidelidades_exato`) |
| `casos` | $(\omega^{(a)},S_a,\bar n_a,\omega^{(b)},S_b,\bar n_b)$ | uma curva por tupla |

## 4. Média sobre $\delta$ — `nos_desvio`

$$\langle f\rangle=\int d\delta\,\frac{e^{-\delta^2/2s^2}}{\sqrt{2\pi}s}f(\delta)\approx\sum_{i=1}^{n_{gh}}\frac{w_i}{\sqrt\pi}\,f\big(\sqrt2\,s\,x_i\big),$$
com $(x_i,w_i)$ os nós e pesos de Gauss-Hermite (`numpy.polynomial.hermite.hermgauss`). Para $s=0$ (sem mola) há um único nó, $\delta=0$, com peso 1.

Com $n_{gh}=24$ a diferença para $n_{gh}=48$ foi $\lesssim2\times10^{-4}$ em $F_1$ no caso mais largo ($\bar n=10$, $s=2.75$). O padrão é 40.

In [ ]:
d, p = c.nos_desvio(0.3, 4.0, 2.0)
print('soma dos pesos =', p.sum(), '   <delta^2> =', np.sum(p * d**2), '   s^2 =', c.largura_desvio(0.3, 4.0, 2.0)**2)

## 5. Um fóton — `Gam`, `t_1foton`, `fase_ref`, `xi_in`, `grade`

Para um desvio fixo $\delta$:
$$\Gamma_\delta(\nu)=\frac\gamma2+i(\Delta+\delta-\nu)\quad(\texttt{Gam}),\qquad
t_\delta(\nu)=1-\frac{\gamma}{\Gamma_\delta(\nu)}=-\frac{\Gamma_\delta^*(\nu)}{\Gamma_\delta(\nu)}\quad(\texttt{t\_1foton}).$$
Para cada $\delta$, $|t_\delta|=1$. Mas a média $\langle t_\delta\rangle$ tem módulo menor que 1, e essa perda de módulo é o efeito da mistura.

**Fóton ideal** (**`fase_ref`**): $\tilde\xi=u\,\xi$, com $u=\langle t_\delta\rangle/|\langle t_\delta\rangle|$ (`'media'`). Para $s=0$ isso dá $-\Gamma^*/\Gamma$, a mesma referência do artigo.

**Pacote** (**`xi_in`**, Eq. (4) do artigo): $\xi(\omega)=(2\pi\sigma^2)^{-1/4}e^{-(\omega-\omega_c)^2/4\sigma^2}$.

**Grade** (**`grade`**): janela $\omega_c\pm k_\sigma\sigma$, espaçamento $\approx\min(\sigma,\gamma)/$`pts_per_width`.

In [ ]:
nu = np.linspace(-15, 15, 600)
for nbar, cor in [(0, 'k'), (2, '#8A2BE2'), (10, '#1E90FF')]:
    d, p = c.nos_desvio(0.3, 4.0, nbar)
    tm = sum(pi * c.t_1foton(di, nu) for di, pi in zip(d, p))
    plt.plot(nu, np.abs(tm)**2, color=cor, label=fr'$|\langle t_\delta\rangle|^2$, $\bar n={nbar}$')
plt.xlabel(r'$\nu$'); plt.legend(); plt.tight_layout(); plt.show()

## 6. Fidelidade — `fidelidades`, `W1`, `W2`

Para $(\delta_a,\delta_b)$ fixos, o sítio é o de Brod-Combes sem mola, mas com os dois átomos dessintonizados de forma diferente:

$$\tau_a=\int d\nu\;\tilde\xi^*(\nu)\,\xi(\nu)\,t_{\delta_a}(\nu)\qquad(\text{idem } b),$$

$$N=-\frac{i\chi\gamma^2}{\pi}\int dE\;K_f(E)\,P(E)\,Q(E),$$
$$P(E)=\int d\nu\,\frac{\xi(\nu)}{\Gamma_{\delta_a}(\nu)}\frac{\xi(E-\nu)}{\Gamma_{\delta_b}(E-\nu)},\qquad
Q(E)=\int d\omega\,\frac{\tilde\xi^*(\omega)}{\Gamma_{\delta_a}(\omega)}\frac{\tilde\xi^*(E-\omega)}{\Gamma_{\delta_b}(E-\omega)},$$
$$K_f(E)=\Big(1+\frac{2i\chi}{\Sigma}\Big)^{-1},\qquad\Sigma=\Gamma_{\delta_a}(\nu_a)+\Gamma_{\delta_b}(\nu_b)=\gamma+i(2\Delta+\delta_a+\delta_b-E).$$

Esse é o termo não linear da matriz $S$ dada, com $g\to0$ e $\Delta\to\Delta+\delta$. A $\delta(\omega_a+\omega_b-\nu_a-\nu_b)$ reduz a integral tripla a convoluções 1D, calculadas por FFT. Isso fica no laço sobre `i` dentro de `fidelidades`: os $n_{gh}$ valores de $\delta_b$ são tratados de uma vez por vetorização.

Como a mola não muda de estado (na aproximação lenta, $x$ é conservado), o vetor de amplitudes é
$$x(\delta_a,\delta_b)=\big(1,\;\tau_b,\;\tau_a,\;e^{i\phi}(\tau_a\tau_b+N)\big)$$
para as componentes $|00\rangle,|01\rangle,|10\rangle,|11\rangle$. A fidelidade com o estado ideal puro, $F=\langle\psi_{\rm id}|\rho|\psi_{\rm id}\rangle$ ([Wikipedia](https://en.wikipedia.org/wiki/Fidelity_of_quantum_states)), com média de Haar, é
$$F_{1,2}=\int d\delta_a\,d\delta_b\;P(\delta_a)P(\delta_b)\;x^\dagger W_{1,2}\,x,$$
$$W_1=\frac{J+I}{20}\ \ (\text{Haar em 2 qubits}),\qquad W_2=\frac{M\otimes M}{36},\ M=\begin{pmatrix}2&1\\1&2\end{pmatrix}\ \ (\text{Haar em estados produto}).$$

As matrizes são as mesmas de `calcular_fock.py` (**`W1`**, **`W2`**). Lá a soma era sobre os estados finais $m$ da mola. Aqui a média é sobre $\delta$. Nos dois casos, cada termo é uma componente do estado misto. Sem mola, a expressão recai nas Eqs. (A2) e (A5) do artigo.

In [ ]:
print('sem mola, sigma=2:', c.fidelidades((1, 0, 0), (1, 0, 0), 2.0), '  (Fig. 5, single site: pico ~0.78)')
for nbar in [0, 2, 10]:
    print(f'nbar={nbar:2d}, sigma=2:', c.fidelidades((0.3, 4.0, nbar), (0.3, 4.0, nbar), 2.0))

## 7. Validação contra o resultado térmico exato — `fidelidades_exato`

Comparação entre a aproximação lenta e a soma exata $\sum p_{n_a}p_{n_b}F_{\rm Fock}(n_a,n_b)$, com o mesmo fóton de referência:

Mola com $\omega=0.3$, $S=1$ (em $\gamma=4.5$, ou seja $\omega/\gamma\approx0.07$):

| $\bar n$ | $\sigma$ | $F_1$ lento | $F_1$ exato | $F_2$ lento | $F_2$ exato | tempo do exato |
|---|---|---|---|---|---|---|
| 0 | 0.1 | 0.4175 | 0.4094 | 0.4553 | 0.4463 | 0.4 s |
| 0 | 1.0 | 0.6696 | 0.6684 | 0.6776 | 0.6764 | 0.6 s |
| 0 | 3.0 | 0.7292 | 0.7293 | 0.7356 | 0.7356 | 0.4 s |
| 0 | 20 | 0.4161 | 0.4161 | 0.4546 | 0.4546 | 0.6 s |
| 1 | 0.1 | 0.4153 | 0.3945 | 0.4536 | 0.4305 | 258 s |
| 1 | 2.0 | 0.74994 | 0.74997 | 0.75703 | 0.75701 | ~4 min |

A aproximação leva ~0,02 s por ponto. O exato com $\bar n=1$ leva ~4 min por ponto e cresce rapidamente com $\bar n$.


* Para $\sigma\gtrsim\omega$ a aproximação concorda com o exato na 3ª casa decimal.
* Para $\sigma<\omega$ ela superestima $F_1$ em 0,01–0,02. Isso é esperado: o pacote estreito resolve as bandas laterais $\pm\omega$, de modo que a troca de quanta com a mola vira um erro espectralmente distinguível, que a imagem estática não captura.
* Quanto menor $\omega/\gamma$ e $\omega/\sigma$, melhor a aproximação.

A célula abaixo refaz um ponto (leva alguns segundos; $\bar n$ maiores ficam lentos rapidamente):

In [ ]:
m = (0.3, 1.0, 0.0)
for s in [0.1, 3.0]:
    print(f'sigma={s}:  lento', c.fidelidades(m, m, s), '   exato', c.fidelidades_exato(m, m, s))

## 8. Varredura, saída e figura — `calcular_caso`, `nome_arquivo`, `main`, `plot_termico.py`

`calcular_caso(w_a, S_a, nbar_a, w_b, S_b, nbar_b)` percorre `n_sigma` valores de $\sigma$, calcula `fidelidades` e salva `data/termico_*.npz` com `sigmas`, `F1`, `F2`, os parâmetros e as larguras `s_a`, `s_b`. `main()` roda todos os `casos`. Para gerar os dados, rode `python3 calcular_termico.py` (≈1 min por caso).

`plot_termico.py` lê os arquivos listados em `curvas` (arquivo, cor, rótulo, estilo) e desenha $F_1(\sigma)$. A linha de $F_2$ está comentada, como em `plot_fock.py`.

**Casos padrão:** $\omega=0.3\ll\gamma=4.5$, $S=4$ ($g=0.3$), $\bar n=0,2,10$, o que dá $s=0.60,\,1.34,\,2.75$.

**Leitura:**
* O caso sem mola reproduz a curva *single site* da Fig. 5.
* A fidelidade cai rapidamente com a temperatura. O pico de $F_1$ passa de 0,78 (sem mola) para ~0,74 ($\bar n=0$), ~0,63 ($\bar n=2$) e ~0,41 ($\bar n=10$). Quando $s$ se aproxima de $\gamma/2$, cada realização de $\delta$ desloca a linha de forma diferente, as fases não lineares de realizações diferentes deixam de se somar coerentemente e a porta perde a fase condicional.
* Para $\bar n=10$, $F_1$ chega a ficar abaixo de 0,4, o valor "sem interação". A mistura reduz $|\langle t_\delta\rangle|$, o que também degrada os ramos de um fóton.

In [ ]:
from IPython.display import Image
Image('fidelidade_sigma_mola_termico.png')